# IPUMS [NHGIS](https://www.nhgis.org) Data Extraction Using [ipumsr](https://cran.r-project.org/web/packages/ipumsr/index.html) - Supplemental Exercise 3
### by [Kate Vavra-Musser](https://vavramusser.github.io) for the [R Spatial Notebook Series](https://vavramusser.github.io/r-spatial)

## Introduction
This notebook provides an additional example of the IPUMS NHGIS data extraction process using the IPUMS API via the ipumsr R package.  This exercise is a supplement to the workflow introducted in Chapter 3.4 IPUMS NHGIS Data Extraction Using ipumsr.

### Notebook Goals
This notebook replicates the IPUMS NHGIS data extraction process and extracts a NHGIS polygon dataset on block group population and accompanying shapefile.  The resulting data file is used in subsequent notebooks in the R Spatial Notebooks series.  The notebook provides an example of extracting spatial data limited by a user-defined extent with attached attribute data from the IPUMS NHGIS repository.

### ✨ Prerequisites ✨
* Complete [Introduction to IPUMS and the IPUMS API](https://platform.i-guide.io/notebooks/82d3b176-e4e6-4307-8186-318a3fe6c81a)
* Set Up Your [IPUMS Account and API Key](https://account.ipums.org/api_keys)
* Complete [Introduction to sf: Reading, Writing, and Inspecting Vector Data](https://platform.i-guide.io/notebooks/9968babe-22e4-4c3d-98e2-d8b45e9672cd)
* Complete [IPUMS NHGIS Data Extraction Using ipumsr](https://platform.i-guide.io/notebooks/be08e56e-1c08-458e-a230-263c64d386bc)
### Notebook Overview
1. Setup
2. Extraction Workflow: Shapefiles Restricted to a Geographic Extent + Tabular Data

---

## 1. Setup
This section will guide you through the process of installing essential packages and setting your IPUMS API key.

#### Required Packages

[**dplyr**](https://cran.r-project.org/web/packages/dplyr/index.html) A Grammar of Data Manipulation. This notebook uses the the following functions from *dplyr*.

* [*filter*](https://rdrr.io/cran/dplyr/man/filter.html) · keep rows that match a condition

[**ipumsr**](https://cran.r-project.org/web/packages/ipumsr/index.html) An R Interface for Downloading, Reading, and Handling IPUMS Data.  This notebook uses the the following functions from *ipumsr*.

* [*define_extract_nhgis*](https://rdrr.io/cran/ipumsr/man/define_extract_nhgis.html) · define an IPUMS NHGIS extract request
* [*download_extract*](https://rdrr.io/cran/ipumsr/man/download_extract.html) · download a completed IPUMS data extract
* [*get_metadata_nhgis*](https://rdrr.io/cran/ipumsr/man/get_metadata_nhgis.html) · list available data sources from IPUMS NHGIS
* [*read_ipums_sf*](https://rdrr.io/cran/ipumsr/man/read_ipums_sf.html) · read spatial data from an IPUMS extract
* [*read_nhgis*](https://rdrr.io/cran/ipumsr/man/read_nhgis.html) · read tabular data from an NHGIS extract
* [*set_ipums_api_key*](https://rdrr.io/cran/ipumsr/man/set_ipums_api_key.html) · set your IPUMS API key
* [*submit_extract*](https://rdrr.io/cran/ipumsr/man/submit_extract.html) · submit an extract request via the IPUMS API
* *tst_spec* · create a *tst_spec* object containing a time series table specification
* [*wait_for_extract*](https://rdrr.io/cran/ipumsr/man/wait_for_extract.html) · wait for an extract to finish processing

[**purrr**](https://cran.r-project.org/web/packages/purrr/index.html) A complete and consistent functional programming toolkit for R. This notebook uses the the following functions from *purrr*.

* [*map()*](https://rdrr.io/cran/purrr/man/map.html) and [*map_dfr()*](https://rdrr.io/cran/purrr/man/map_dfr.html) · apply a function to each element of a vector

[**sf**](https://cran.r-project.org/web/packages/sf/index.html) Support for simple features, a standardized way to encode spatial vector data. Binds to 'GDAL' for reading and writing data, to 'GEOS' for geometrical operations, and to 'PROJ' for projection conversions and datum transformations. Uses by default the 's2' package for spherical geometry operations on ellipsoidal (long/lat) coordinates.  This notebook uses the following functions from *sf*.

* [*st_write*](https://rdrr.io/cran/sf/man/st_write.html) · Write simple features object to file or database

### 1a. Install and Load Required Packages
If you have not already installed the required packages, uncomment and run the code below:

In [1]:
# install.packages(c("dplyr", "ipumsr", "purr", "sf"))  

Load the packages into your workspace.

In [2]:
library(dplyr)
library(ipumsr)
library(purrr)
library(sf)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Linking to GEOS 3.9.1, GDAL 3.2.1, PROJ 7.2.1; sf_use_s2() is TRUE



### 1b. Set Your IPUMS API Key

Store your [IPUMS API key](https://account.ipums.org/api_keys) in your environment using the following code.

Refer to [Chapter 1.1 Introduction to IPUMS and the IPUMS API](https://platform.i-guide.io/notebooks/82d3b176-e4e6-4307-8186-318a3fe6c81a) for instructions on setting up your IPUMS account and API key.

In [3]:
ipumps_api_key = readline("Please enter your IPUMS API key: ")
set_ipums_api_key(ipumps_api_key, save = T, overwrite = T)

Existing .Renviron file copied to C:\Users\vavra\Documents/.Renviron_backup for backup purposes.

The environment variable IPUMS_API_KEY has been set and saved for future sessions.



## 2. NHGIS Time-Series + Polygons with Geographic Extent - Method 1

### 2a. View and Filter the List of Geography Shapefiles

Forthis exercise, we will again work with the total population time-series table *CL8* that we worked with in Chapter 2.4 and Chapte 2.4a.  Therefore, we will jump directly into e will taking a look at the list of geography shapefiles that fit our critera.  We are looking for block group boundary shapefiles but want to restrict the extent of our query to onl the state of Minnesota.  Therefore, let's filter the shapefile metadata to only incluede shapefiles which include the word "block group" on the description of their *geographic_level* and have *Minnesota* as their extent.  We will also focus on only shapefiles using the 2010 Tiger-Line files so we will also filter based on the *year = 2010* criteria.

In [4]:
metadata_shp <- get_metadata_nhgis("shapefiles") %>%
    filter(year == 2010, extent == "Minnesota", grepl("block group", geographic_level, ignore.case = T)) %>%
    print(n = Inf)

Warning message:
"`get_metadata_nhgis()` was deprecated in ipumsr 0.9.0.
i Please use `get_metadata_catalog()` to obtain summary metadata or
  `get_metadata()` to obtain detailed metadata."


# A tibble: 2 x 6
  name                     year  geographic_level extent    basis       sequence
  <chr>                    <chr> <chr>            <chr>     <chr>          <int>
1 270_blck_grp_2010_tl2010 2010  Block Group      Minnesota 2010 TIGER~      679
2 270_blck_grp_2010_tl2020 2010  Block Group      Minnesota 2020 TIGER~      731


This filter resulted in a list of two potential shapefiles.  Let's select the 2010 shapefile based on 2010 Tiger-Line shapefiles for states (*270_blck_grp_2010_tl2010*).

### 2b. Shapefile Extraction Specification and Submission

Now that we have selected our shapefile (*270_blck_grp_2010_tl2010*) we are ready to define and submit our extraction request to the IPUMS API.  We already know we want to use time-series table *CL8* at the block group (*blk_grp*) geographic level.

In [5]:
extract_definition <- define_extract_nhgis(description = "I-GUIDE NHGIS County Population  Extraction",
                                           time_series_tables = tst_spec(name = "CL8",
                                                                         geog_levels = "blck_grp"),
                                           shapefiles = "270_blck_grp_2010_tl2010")

Warning message:
"`define_extract_nhgis()` was deprecated in ipumsr 0.9.0.
i Please use `define_extract_agg()` instead."


Submitting the extraction definition object *extract_definition* to the API.

In [6]:
extraction_submitted <- submit_extract(extract_definition)
extraction_complete <- wait_for_extract(extraction_submitted)
extraction_complete$status
filepath <- download_extract(extraction_submitted, overwrite = T)

Successfully submitted IPUMS NHGIS extract number 125

Checking extract status...

Waiting 10 seconds...

Checking extract status...

IPUMS NHGIS extract 125 is ready to download.



[1] "completed"

  |======================================================================| 100%
  |======================================================================| 100%


Data file saved to C:/Users/vavra/Dropbox/R Spatial/r-spatial/03 IPUMS Data Acquisition and Extraction/nhgis0125_csv.zip
Shapefile saved to C:/Users/vavra/Dropbox/R Spatial/r-spatial/03 IPUMS Data Acquisition and Extraction/nhgis0125_shape.zip



The result of the extraction request will be two files 1) a time-series table containing the populationd data and 2) the places (points) geography shapefile.  The next step is to read these two files into R.

In [7]:
dat <- read_nhgis(filepath[1])
shp <- read_ipums_sf(filepath[2])

Warning message:
"`read_nhgis()` was deprecated in ipumsr 0.9.0.
i Please use `read_ipums_agg()` instead."
Use of data from IPUMS NHGIS is subject to conditions including that users should cite the data appropriately. Use command `ipums_conditions()` for more details.

Rows: 217740 Columns: 18
-- Column specification --------------------------------------------------------
Delimiter: ","
chr  (6): GISJOIN, STATE, STATEA, COUNTY, COUNTYA, TRACTA
dbl (12): GEOGYEAR, BLCK_GRPA, CL8AA1990, CL8AA1990L, CL8AA1990U, CL8AA2000,...

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


Let's take a look at the dimesions of the time-series data (*dat*) and the shapefile (*shp*).

In [8]:
dim(dat)
dim(shp)

[1] 217740     18

[1] 4108   16

The time-series table includes 18 variables for 217,740 block groups in the entire United States and the shapefile includes 16 attributes for the 4,108 block groups in the state of Minnesota.

Let's take a look at the first few lines of the time-series table on population and the block groups shapefile.

In [9]:
head(dat)

GISJOIN,GEOGYEAR,STATE,STATEA,COUNTY,COUNTYA,TRACTA,BLCK_GRPA,CL8AA1990,CL8AA1990L,CL8AA1990U,CL8AA2000,CL8AA2000L,CL8AA2000U,CL8AA2010,CL8AA2020,CL8AA2020L,CL8AA2020U
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
G01000100201001,2010,Alabama,01,Autauga County,001,020100,1,731.67,709,732,748.02,728,749,698,575,515,575
G01000100201002,2010,Alabama,01,Autauga County,001,020100,2,1041.00,1041,1041,1172.00,1172,1172,1214,1200,1200,1200
G01000100202001,2010,Alabama,01,Autauga County,001,020200,1,850.00,850,850,857.00,857,857,1003,974,974,974
G01000100202002,2010,Alabama,01,Autauga County,001,020200,2,1181.00,1181,1181,1035.00,1035,1035,1167,1081,1081,1141
G01000100203001,2010,Alabama,01,Autauga County,001,020300,1,1996.00,1996,1996,2445.00,2445,2445,2549,2377,2377,2377
G01000100203002,2010,Alabama,01,Autauga County,001,020300,2,956.00,956,956,894.00,894,894,824,839,839,839


In [10]:
head(shp)

Registered S3 method overwritten by 'geojsonsf':
  method        from   
  print.geojson geojson



STATEFP10,COUNTYFP10,TRACTCE10,BLKGRPCE10,GEOID10,NAMELSAD10,MTFCC10,FUNCSTAT10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,GISJOIN,Shape_area,Shape_len,geometry
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<MULTIPOLYGON [m]>
27,005,450800,1,270054508001,Block Group 1,G5030,S,223817614,19858337,+46.8545174,-095.9870930,G27000504508001,243675948.2,91686.074,MULTIPOLYGON (((5950.534 10...
27,139,080903,2,271390809032,Block Group 2,G5030,S,2015945,1676070,+44.7105203,-093.4493207,G27013900809032,3692016.7,9869.906,MULTIPOLYGON (((202707.8 81...
27,123,041200,3,271230412003,Block Group 3,G5030,S,558985,1891,+45.0619148,-093.2013428,G27012300412003,560876.2,3961.686,MULTIPOLYGON (((220402.9 84...
27,123,036600,2,271230366002,Block Group 2,G5030,S,451240,0,+44.9192490,-093.1533467,G27012300366002,451242.1,3416.141,MULTIPOLYGON (((224771.1 83...
27,123,042101,1,271230421011,Block Group 1,G5030,S,2553064,308889,+45.0340692,-093.0962124,G27012300421011,2861954.1,7607.575,MULTIPOLYGON (((229510.9 84...
27,123,034701,1,271230347011,Block Group 1,G5030,S,561389,0,+44.9599320,-093.0253077,G27012300347011,561386.3,3004.427,MULTIPOLYGON (((234750.5 83...


## 2. NHGIS Time-Series + Polygons with Geographic Extent - Method 2

Alternately, we could have set up our querty using the block group shapefile for the entire United States (*us_blck_grp_2010_tl2010*) and then specified the geographic extent in the extraction definition step.

In [11]:
extract_definition <- define_extract_nhgis(description = "I-GUIDE NHGIS County Population  Extraction",
                                           time_series_tables = tst_spec(name = "CL8",
                                                                         geog_levels = "blck_grp"),
                                           shapefiles = "us_blck_grp_2010_tl2010",
                                           geographic_extents = c("Minnesota"))

Submitting the extraction definition object *extract_definition* to the API.

In [12]:
extraction_submitted <- submit_extract(extract_definition)
extraction_complete <- wait_for_extract(extraction_submitted)
extraction_complete$status
filepath <- download_extract(extraction_submitted, overwrite = T)

ERROR: [1m[33mError[39m in `ipums_api_request()`:[22m
[33m![39m API request failed with status 400.
[31mx[39m Geographic extents invalid: Minnesota


Again, the result of the extraction request will be two files 1) a time-series table containing the populationd data and 2) the places (points) geography shapefile.  The next step is to read these two files into R.

In [ ]:
dat <- read_nhgis(filepath[1])
shp <- read_ipums_sf(filepath[2])

Use of data from IPUMS NHGIS is subject to conditions including that users should cite the data appropriately. Use command `ipums_conditions()` for more details.

Rows: 217740 Columns: 18
-- Column specification --------------------------------------------------------
Delimiter: ","
chr  (6): GISJOIN, STATE, STATEA, COUNTY, COUNTYA, TRACTA
dbl (12): GEOGYEAR, BLCK_GRPA, CL8AA1990, CL8AA1990L, CL8AA1990U, CL8AA2000,...

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


If we take a look at the dimesions of the time-series data (*dat*) and the shapefile (*shp*) and at the first few lines of each file, we can see that this method of setting up the data extraction resulted in the same files as the first method.

In [ ]:
dim(dat)
dim(shp)

[1] 217740     18

[1] 219774     16

In [ ]:
head(dat)

GISJOIN,GEOGYEAR,STATE,STATEA,COUNTY,COUNTYA,TRACTA,BLCK_GRPA,CL8AA1990,CL8AA1990L,CL8AA1990U,CL8AA2000,CL8AA2000L,CL8AA2000U,CL8AA2010,CL8AA2020,CL8AA2020L,CL8AA2020U
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
G01000100201001,2010,Alabama,01,Autauga County,001,020100,1,731.67,709,732,748.02,728,749,698,575,515,575
G01000100201002,2010,Alabama,01,Autauga County,001,020100,2,1041.00,1041,1041,1172.00,1172,1172,1214,1200,1200,1200
G01000100202001,2010,Alabama,01,Autauga County,001,020200,1,850.00,850,850,857.00,857,857,1003,974,974,974
G01000100202002,2010,Alabama,01,Autauga County,001,020200,2,1181.00,1181,1181,1035.00,1035,1035,1167,1081,1081,1141
G01000100203001,2010,Alabama,01,Autauga County,001,020300,1,1996.00,1996,1996,2445.00,2445,2445,2549,2377,2377,2377
G01000100203002,2010,Alabama,01,Autauga County,001,020300,2,956.00,956,956,894.00,894,894,824,839,839,839


In [ ]:
head(shp)

ERROR while rich displaying an object: Error in loadNamespace(x): there is no package called 'geojsonio'

Traceback:
1. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
2. tryCatchList(expr, classes, parentenv, handlers)
3. tryCatchOne(expr, names, parentenv, handlers[[1L]])
4. doTryCatch(return(expr), name, parentenv, handler)
5. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler)
6. repr::mime2repr[[mime]](obj)
7. repr_geojson.sf(obj)
8. repr_geojson(geo

STATEFP10,COUNTYFP10,TRACTCE10,BLKGRPCE10,GEOID10,NAMELSAD10,MTFCC10,FUNCSTAT10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,GISJOIN,Shape_area,Shape_len,geometry
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<MULTIPOLYGON [m]>
06,071,000301,3,060710003013,Block Group 3,G5030,S,163855,0,+34.0658794,-117.7026743,G06007100003013,163854.8,2009.936,MULTIPOLYGON (((-1970669 -1...
06,071,000403,2,060710004032,Block Group 2,G5030,S,970863,0,+34.0370965,-117.7090928,G06007100004032,970863.1,4022.155,MULTIPOLYGON (((-1971712 -1...
06,071,009800,3,060710098003,Block Group 3,G5030,S,993825,0,+34.5385341,-117.3046308,G06007100098003,993825.1,4440.150,MULTIPOLYGON (((-1923418 -1...
06,065,042716,2,060650427162,Block Group 2,G5030,S,998386,0,+33.6869386,-117.2477697,G06006500427162,998386.4,6155.236,MULTIPOLYGON (((-1939239 -2...
06,065,042720,1,060650427201,Block Group 1,G5030,S,16490415,0,+33.7732688,-117.0988159,G06006500427201,16490413.1,22261.412,MULTIPOLYGON (((-1922602 -2...
06,071,000201,1,060710002011,Block Group 1,G5030,S,175013,0,+34.0835725,-117.7066024,G06007100002011,175014.2,1727.278,MULTIPOLYGON (((-1970663 -1...


Both files have the common column *GISJOIN* which should be familiar since we used it to merge the time-series and shapefile files in Chapter 2.4.  This time, we will not merge the files here but will instead save them seperately as a comma-seperated values (.csv) file for the time-series data and as a shapefile (.shp) for the geographic data.

Before we do that however, we should remove a few columns that require a lot of memory.  This will make it easier to save and work work this data.

In [ ]:
# merge the time-series population data with the county geographic data
dat_shp <- merge(dat, shp, by = "GISJOIN")

In [ ]:
dat_shp <- dat_shp[, !names(dat_shp) %in% c("ALAND10", "AWATER10", "Shape_area")]

We are ready to save the files to our workspace.

In [ ]:
st_write(dat_shp, "ipums_nhgis_blockgroups.shp", driver = "ESRI Shapefile", delete_dsn = T)

Warning message in CPL_write_ogr(obj, dsn, layer, driver, as.character(dataset_options), :
"GDAL Error 1: ipums_nhgis_blockgroups.shp does not appear to be a file or directory."


Deleting source `ipums_nhgis_blockgroups.shp' failed
Writing layer `ipums_nhgis_blockgroups' to data source 
  `ipums_nhgis_blockgroups.shp' using driver `ESRI Shapefile'
Writing 217221 features with 29 fields and geometry type Multi Polygon.


At the end of this notebook we have saved a copy of the time-series table with population data from the 1990, 2000, 2010, and 2020 Decennial Censuses for block groups in the United States to the file *ipums_nhgis_blockgroups.csv* and a copy of the complementary geographic data file for block groups in the state of Minnesota to the shapefile *ipums_nhgis_blockgroups.shp*.